In [ ]:
import pandas as pd
import numpy as np
import json
import joblib
import matplotlib.pyplot as plt
from pathlib import Path
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

PROCESSED_DIR = Path('processed_data')

print('Setup selesai.')

In [ ]:
X_train = pd.read_csv(PROCESSED_DIR / 'X_train.csv')
X_test  = pd.read_csv(PROCESSED_DIR / 'X_test.csv')
y_train = pd.read_csv(PROCESSED_DIR / 'y_train.csv').squeeze('columns')
y_test  = pd.read_csv(PROCESSED_DIR / 'y_test.csv').squeeze('columns')

with open(PROCESSED_DIR / 'feature_columns.json') as f:
    feature_cols = json.load(f)['feature_columns']

X_train = X_train[feature_cols]
X_test  = X_test[feature_cols]

print(f'Train: {X_train.shape[0]} rows')
print(f'Test:  {X_test.shape[0]} rows')
print(f'Features: {len(feature_cols)}')

In [ ]:
class WalkForwardSplit:
    def __init__(self, n_splits=5, initial_frac=0.7, step_frac=0.06):
        self.n_splits = n_splits
        self.initial_frac = initial_frac
        self.step_frac = step_frac

    def split(self, X, y=None, groups=None):
        n = len(X)
        for i in range(self.n_splits):
            train_end = int(n * (self.initial_frac + i * self.step_frac))
            test_end  = int(n * (self.initial_frac + (i + 1) * self.step_frac))
            train_idx = np.arange(train_end)
            test_idx  = np.arange(train_end, test_end)
            yield train_idx, test_idx

    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits

wf = WalkForwardSplit(n_splits=5, initial_frac=0.7, step_frac=0.06)

print('WalkForwardSplit siap (5 fold, expanding window).')
for i, (tr, te) in enumerate(wf.split(X_train), 1):
    print(f'  Fold {i}: train [0:{len(tr)}], test [{tr[-1]}:{te[-1]}]')

In [ ]:
import math
from itertools import product

param_grid = {
    'n_estimators': [300, 500, 800],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0],
    'min_child_weight': [1, 3, 5],
}

n_combo = len(list(product(*param_grid.values())))
n_fits = n_combo * wf.n_splits
print(f'Grid: {n_combo} kombinasi params x {wf.n_splits} fold = {n_fits} total fit')
print()

model = XGBRegressor(random_state=42, n_jobs=-1, verbosity=0)

gs = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=wf,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=2,
)

gs.fit(X_train, y_train)

print(f'\n=== GridSearch Selesai ===')
print(f'Best params: {gs.best_params_}')
print(f'Best CV MAE: {-gs.best_score_:.4f}')

In [ ]:
best_model = XGBRegressor(**gs.best_params_, random_state=42, n_jobs=-1, verbosity=0)
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mape = np.mean(np.abs((y_test - y_pred) / (y_test + 1e-6))) * 100
r2   = r2_score(y_test, y_pred)

print('=== Hasil Evaluasi Test Set ===')
print(f'MAE  : {mae:.4f}')
print(f'RMSE : {rmse:.4f}')
print(f'MAPE : {mape:.2f}%')
print(f'R²   : {r2:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].plot(y_test.values, label='Actual', color='steelblue', linewidth=0.8)
axes[0].plot(y_pred, label='Predicted', color='orangered', linewidth=0.8, alpha=0.7)
axes[0].set_title('Actual vs Predicted (Test Set)')
axes[0].set_xlabel('Sample')
axes[0].set_ylabel('PM2.5')
axes[0].legend()

axes[1].scatter(y_test, y_pred, alpha=0.4, s=15, color='steelblue')
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', alpha=0.5)
axes[1].set_xlabel('Actual')
axes[1].set_ylabel('Predicted')
axes[1].set_title(f'Scatter (R² = {r2:.4f})')

residuals = y_test - y_pred
axes[2].scatter(y_pred, residuals, alpha=0.4, s=15, color='teal')
axes[2].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[2].set_xlabel('Predicted')
axes[2].set_ylabel('Residual')
axes[2].set_title('Residual Plot')

plt.tight_layout()
plt.show()

In [ ]:
importance = best_model.feature_importances_
feat_df = pd.DataFrame({'feature': feature_cols, 'importance': importance})
feat_df = feat_df.sort_values('importance', ascending=True)

top_n = min(20, len(feat_df))
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(range(top_n), feat_df['importance'].iloc[-top_n:], color='steelblue')
ax.set_yticks(range(top_n))
ax.set_yticklabels(feat_df['feature'].iloc[-top_n:])
ax.set_xlabel('Importance')
ax.set_title(f'Top {top_n} Feature Importance')
plt.tight_layout()
plt.show()

In [ ]:
best_model_path = PROCESSED_DIR / 'best_model.joblib'
joblib.dump(best_model, best_model_path)

report = {
    'best_params': gs.best_params_,
    'cv_best_mae': round(-gs.best_score_, 4),
    'test_metrics': {
        'mae': round(mae, 4),
        'rmse': round(rmse, 4),
        'mape': round(mape, 2),
        'r2': round(r2, 4),
    },
    'n_features': len(feature_cols),
    'n_train': len(X_train),
    'n_test': len(X_test),
}

with open(PROCESSED_DIR / 'modelling_report.json', 'w') as f:
    json.dump(report, f, indent=2)

print('Saved:')
print(f'  {best_model_path.name}')
print(f'  modelling_report.json')

In [ ]:
import joblib
from pathlib import Path
from datetime import timedelta

PROCESSED_DIR = Path('processed_data')

model = joblib.load(PROCESSED_DIR / 'best_model.joblib')
with open(PROCESSED_DIR / 'feature_columns.json') as f:
    feature_cols = json.load(f)['feature_columns']

seg_files = sorted(PROCESSED_DIR.glob('segmen_*.csv'))
hist = pd.read_csv(seg_files[-1])
hist['datetime'] = pd.to_datetime(hist['datetime'])
hist = hist.sort_values('datetime').tail(168).reset_index(drop=True)

def _build_row(h, target_dt):
    h = h.sort_values('datetime').reset_index(drop=True)
    hr = target_dt.hour
    dow = target_dt.weekday()

    def lv(col, default=0.0):
        v = pd.to_numeric(h[col], errors='coerce').dropna()
        return float(v.iloc[-1]) if len(v) > 0 else default

    def lg(col, lag, default=0.0):
        if len(h) < lag:
            return lv(col, default)
        v = pd.to_numeric(h[col], errors='coerce').iloc[-lag]
        return float(v) if not pd.isna(v) else lv(col, default)

    row = {
        'pm1': lv('pm1'),
        'relativehumidity': lv('relativehumidity'),
        'temperature': lv('temperature'),
        'um003': lv('um003'),
        'pm1_lag_1h': lg('pm1', 1),
        'um003_lag_1h': lg('um003', 1),
        'relativehumidity_lag_6h': lg('relativehumidity', 6),
        'relativehumidity_lag_12h': lg('relativehumidity', 12),
        'relativehumidity_lag_24h': lg('relativehumidity', 24),
        'temperature_lag_6h': lg('temperature', 6),
        'temperature_lag_12h': lg('temperature', 12),
        'temperature_lag_24h': lg('temperature', 24),
        'pm25_lag_1h': lg('pm25', 1),
        'pm25_lag_2h': lg('pm25', 2),
        'pm25_lag_3h': lg('pm25', 3),
        'pm25_lag_4h': lg('pm25', 4),
        'pm25_lag_12h': lg('pm25', 12),
        'pm25_lag_24h': lg('pm25', 24),
        'pm25_lag_48h': lg('pm25', 48),
        'pm25_lag_72h': lg('pm25', 72),
        'pm25_lag_124h': lg('pm25', 124),
        'hour': hr,
        'day_of_week': dow,
        'is_weekend': int(dow >= 5),
        'hour_sin': np.sin(2 * np.pi * hr / 24),
        'hour_cos': np.cos(2 * np.pi * hr / 24),
    }

    vals = pd.to_numeric(h['pm25'], errors='coerce')
    for w in [24, 48, 72]:
        tail = vals.dropna().tail(w)
        row[f'pm25_rolling_mean_{w}h'] = float(tail.mean()) if len(tail) > 0 else 0.0

    clean_vals = vals.dropna()
    if len(clean_vals) >= 2:
        row['pm25_diff_1h'] = float(clean_vals.iloc[-1] - clean_vals.iloc[-2])
    else:
        row['pm25_diff_1h'] = 0.0
    if len(clean_vals) >= 25:
        row['pm25_diff_24h'] = float(clean_vals.iloc[-1] - clean_vals.iloc[-25])
    else:
        row['pm25_diff_24h'] = 0.0

    return pd.DataFrame([[row.get(c, 0.0) for c in feature_cols]], columns=feature_cols)

base_time = hist['datetime'].iloc[-1]
preds = []

for step in range(1, 25):
    t = base_time + timedelta(hours=step)
    feat = _build_row(hist, t)
    pm25_pred = max(0.0, float(model.predict(feat)[0]))
    preds.append({'hour': step, 'time': t, 'pm25': round(pm25_pred, 2)})
    new_row = {'datetime': t, 'pm25': pm25_pred}
    for c in ['pm1', 'relativehumidity', 'temperature', 'um003']:
        v = pd.to_numeric(hist[c], errors='coerce').dropna()
        new_row[c] = float(v.iloc[-1]) if len(v) > 0 else 0.0
    hist = pd.concat([hist, pd.DataFrame([new_row])], ignore_index=True)

pred_df = pd.DataFrame(preds)
print('=== Prediksi 24 Jam (Recursive Forecasting) ===')
print(pred_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(pred_df['hour'], pred_df['pm25'], marker='o', color='orangered', linewidth=2)
for _, r in pred_df.iterrows():
    ax.text(r['hour'], r['pm25'] + 0.5, str(r['pm25']), ha='center', fontsize=8)
ax.set_xlabel('Jam ke-')
ax.set_ylabel('PM2.5 Prediksi')
ax.set_title('Prediksi 24 Jam ke Depan (Recursive Forecasting)')
ax.set_xticks(range(1, 25))
plt.tight_layout()
plt.show()